# Notebook 15: Synchronized Data Art — India Breathes

**One Sensor, One Year — India Breathes**

A synchronized animated visualization where multiple art forms breathe together through India's 2024 grid year. One shared clock ticks Jan 1 → Dec 31 — the radial bloom grows, the map pulses green during monsoon, the heartbeat strip scrolls, the heatmap fills in.

**Interaction:** `Play` button auto-advances through 52 weeks. Slider for manual scrub. All panels update in sync via `ipywidgets` observers driving `plotly.FigureWidget` instances.

---
## Section 0: Setup & Data Loading

In [23]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
import ipywidgets as widgets
from IPython.display import display, HTML
import json
import warnings
warnings.filterwarnings('ignore')

# ---------- Load primary data ----------
df = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])

fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
for col in fuel_cols:
    df[col] = df[col].fillna(0)

df['wind_solar'] = df['wind'] + df['solar']
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['dow'] = df['date'].dt.dayofweek
df['day_of_year'] = df['date'].dt.dayofyear

n_days = len(df)

# ---------- Load POSOCO + GeoJSON for map ----------
posoco = pd.read_csv('../data/raw/POSOCO_data.csv')
posoco = posoco[(posoco['yyyymmdd'] >= 20240101) & (posoco['yyyymmdd'] <= 20241231)].copy()
posoco['date'] = pd.to_datetime(posoco['yyyymmdd'], format='%Y%m%d')
posoco['month'] = posoco['date'].dt.month

with open('../data/raw/india_states.geojson') as f:
    india_geo = json.load(f)

# ---------- Palettes ----------
earth_sky = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD',
}

fossil_colors = ['#922B21', '#C0392B', '#E74C3C']
clean_colors = ['#2EC4B6', '#1B4F72', '#85C1E9', '#F5B041', '#A569BD']

BG = '#FAFAF5'
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# ---------- Map settings ----------
INDIA_GEO = dict(
    visible=False, bgcolor=BG,
    lonaxis_range=[68, 98], lataxis_range=[6, 38],
)

# ---------- Load temperature data ----------
df_temp = pd.read_csv('../data/processed/india_2024_temperature.csv', parse_dates=['date'])

print(f'Data ready: {n_days} days, POSOCO: {len(posoco)} days, GeoJSON loaded, Temp: {len(df_temp)} days')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Temperature: {df_temp["india_mean"].min():.1f}°C — {df_temp["india_mean"].max():.1f}°C')

Data ready: 366 days, POSOCO: 366 days, GeoJSON loaded, Temp: 366 days
Date range: 2024-01-01 → 2024-12-31
Temperature: 18.6°C — 32.5°C


---
## Section 1: Precompute All Frame Data

Precompute everything upfront so the animation callbacks are just array swaps — no computation during playback. The slider has **52 positions** (weeks). Each viz maps weeks to its native resolution.

In [24]:
N_WEEKS = 52

# ===== BLOOM DATA =====
# Angles and dimensions for 366 radial bars
angles_deg = np.linspace(0, 360, n_days, endpoint=False)
width_deg = 360 / n_days * 0.95
max_total = df['total'].max()
bloom_heights = df['total'].values / max_total * 5

# Color each bar by clean_share (RdYlGn)
clean_pct = df['clean_share'].values
vmin, vmax = clean_pct.min(), clean_pct.max()
norms = [(p - vmin) / (vmax - vmin) for p in clean_pct]
bloom_colors = sample_colorscale('RdYlGn', norms)

# Precompute 52 frames of r-values (progressive reveal)
bloom_frames = np.zeros((N_WEEKS, n_days))
for w in range(N_WEEKS):
    day_end = min((w + 1) * 7, n_days)
    bloom_frames[w, :day_end] = bloom_heights[:day_end]

# ===== MAP DATA =====
# State-region mapping (from notebook 14)
state_to_region = {
    'NCT of Delhi': 'NR', 'Uttar Pradesh': 'NR', 'Punjab': 'NR',
    'Haryana': 'NR', 'Rajasthan': 'NR', 'Himachal Pradesh': 'NR',
    'Uttarakhand': 'NR', 'Jammu & Kashmir': 'NR', 'Ladakh': 'NR',
    'Chandigarh': 'NR',
    'Maharashtra': 'WR', 'Gujarat': 'WR', 'Madhya Pradesh': 'WR',
    'Chhattisgarh': 'WR', 'Goa': 'WR',
    'Dadra and Nagar Haveli and Daman and Diu': 'WR',
    'Tamil Nadu': 'SR', 'Karnataka': 'SR', 'Kerala': 'SR',
    'Andhra Pradesh': 'SR', 'Telangana': 'SR', 'Puducherry': 'SR',
    'Lakshadweep': 'SR',
    'West Bengal': 'ER', 'Bihar': 'ER', 'Jharkhand': 'ER',
    'Odisha': 'ER', 'Sikkim': 'ER', 'Andaman and Nicobar Islands': 'ER',
    'Assam': 'NER', 'Arunachal Pradesh': 'NER', 'Meghalaya': 'NER',
    'Manipur': 'NER', 'Mizoram': 'NER', 'Nagaland': 'NER', 'Tripura': 'NER',
}

REGIONS = ['NR', 'WR', 'SR', 'ER', 'NER']
map_states = sorted(state_to_region.keys())

# Precompute monthly clean_pct per region
region_monthly_clean = {}
for m in range(1, 13):
    mdf = posoco[posoco['month'] == m]
    for r in REGIONS:
        total = mdf[f'{r}: Total'].sum()
        coal = mdf[f'{r}: Coal'].sum()
        hydro = mdf[f'{r}: Hydro'].sum()
        nuclear = mdf[f'{r}: Nuclear'].sum()
        res = mdf[f'{r}: RES'].sum()
        clean = hydro + nuclear + res
        region_monthly_clean[(r, m)] = 100 * clean / total if total > 0 else 0

# Build 12 months × n_states array for fast lookup
# map_z_frames[month_idx] = array of clean_pct for each state
map_z_frames = np.zeros((12, len(map_states)))
for i, state in enumerate(map_states):
    region = state_to_region[state]
    for m in range(12):
        map_z_frames[m, i] = region_monthly_clean[(region, m + 1)]

# Week-to-month mapping
def week_to_month(w):
    """Convert week index (0-51) to month index (0-11)."""
    day = min((w + 1) * 7, 366)
    return min(11, (day - 1) // 31)

# ===== HEARTBEAT DATA =====
fossil_fuels = ['coal', 'lignite', 'gas']
clean_fuels = ['nuclear', 'hydro', 'wind', 'solar', 'other_re']

# Stacked arrays for mirrored heartbeat
fossil_stack = np.zeros((n_days, len(fossil_fuels) + 1))
for i, fuel in enumerate(fossil_fuels):
    fossil_stack[:, i+1] = fossil_stack[:, i] + df[fuel].values

clean_stack = np.zeros((n_days, len(clean_fuels) + 1))
for i, fuel in enumerate(clean_fuels):
    clean_stack[:, i+1] = clean_stack[:, i] + df[fuel].values

# ===== HEATMAP DATA =====
dow_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
pivot_heat = df.pivot_table(values='total', index='week', columns='dow', aggfunc='mean')
pivot_heat.columns = dow_order
all_weeks = sorted(pivot_heat.index)
heat_z_full = pivot_heat.values
heat_z_min = np.nanmin(heat_z_full)
heat_z_max = np.nanmax(heat_z_full)

# Precompute 52 frames of heatmap z-values (progressive reveal)
heat_frames = []
for w in range(N_WEEKS):
    z = np.full_like(heat_z_full, np.nan, dtype=float)
    w_end = min(w + 1, len(all_weeks))
    z[:w_end, :] = heat_z_full[:w_end, :]
    heat_frames.append(z.copy())

# ===== DATE LABELS =====
from datetime import datetime, timedelta
base_date = datetime(2024, 1, 1)
week_dates = [base_date + timedelta(days=w*7) for w in range(N_WEEKS)]
week_labels = [d.strftime('%b %d') for d in week_dates]

print(f'Precomputed:')
print(f'  Bloom: {bloom_frames.shape} (weeks × days)')
print(f'  Map: {map_z_frames.shape} (months × states)')
print(f'  Heartbeat: fossil {fossil_stack.shape}, clean {clean_stack.shape}')
print(f'  Heatmap: {len(heat_frames)} frames of {heat_z_full.shape}')
print(f'  Week labels: {week_labels[0]} → {week_labels[-1]}')

# ===== CO2 EMISSIONS DATA =====
co2_daily = df['co2_total'].values  # kt/day
co2_cumulative = np.cumsum(co2_daily) / 1000  # Convert kt to Mt
emissions_intensity = df['emissions_intensity'].values  # tCO2/GWh

# ===== TEMPERATURE DATA =====
temp_daily = df_temp['india_mean'].values  # °C (5-city average)
temp_delhi = df_temp['Delhi_mean'].values
temp_mumbai = df_temp['Mumbai_mean'].values

print(f'  CO2: {co2_daily.sum()/1000:.0f} Mt total, intensity {emissions_intensity.mean():.0f} tCO2/GWh avg')
print(f'  Temperature: {temp_daily.min():.1f}°C — {temp_daily.max():.1f}°C')

Precomputed:
  Bloom: (52, 366) (weeks × days)
  Map: (12, 36) (months × states)
  Heartbeat: fossil (366, 4), clean (366, 6)
  Heatmap: 52 frames of (52, 7)
  Week labels: Jan 01 → Dec 23
  CO2: 1278 Mt total, intensity 721 tCO2/GWh avg
  Temperature: 18.6°C — 32.5°C


---
## Section 2: Build Individual FigureWidgets

Each function returns a `plotly.graph_objects.FigureWidget` — a live, kernel-connected figure that supports in-place data updates without full re-renders.

In [25]:
def build_bloom():
    """Radial bloom — 366 bars in polar plot, color = clean energy %."""
    # Start with all bars at r=0 (will be revealed progressively)
    fig = go.FigureWidget(
        data=[go.Barpolar(
            r=bloom_frames[0],
            theta=angles_deg,
            width=[width_deg] * n_days,
            marker_color=bloom_colors,
            marker_line_width=0,
            base=1.0,
            opacity=0.9,
        )]
    )
    
    # Month labels around the circle
    month_starts = [0, 31, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
    annotations = []
    for day, label in zip(month_starts, month_names):
        angle_rad = (day / n_days) * 2 * np.pi
        x = 0.5 + 0.45 * np.sin(angle_rad)
        y = 0.5 + 0.45 * np.cos(angle_rad)
        annotations.append(dict(
            x=x, y=y, xref='paper', yref='paper',
            text=f'<b>{label}</b>', showarrow=False,
            font=dict(size=10, color='#666'),
        ))
    
    fig.update_layout(
        title=dict(text='Radial Bloom — Clean Energy Share', font=dict(size=14), x=0.5),
        polar=dict(
            radialaxis=dict(visible=False, range=[0, 7]),
            angularaxis=dict(visible=False, direction='clockwise', rotation=90),
            bgcolor=BG,
        ),
        plot_bgcolor=BG, paper_bgcolor=BG,
        height=500, width=500,
        showlegend=False,
        annotations=annotations,
        margin=dict(l=20, r=20, t=50, b=20),
    )
    return fig

def build_map():
    """India choropleth — states colored by clean energy share, updates monthly."""
    fig = go.FigureWidget(
        data=[go.Choropleth(
            geojson=india_geo,
            locations=map_states,
            featureidkey='properties.ST_NM',
            z=map_z_frames[0],
            colorscale='RdYlGn',
            zmin=0, zmax=80,
            colorbar=dict(title='Clean %', len=0.6, thickness=15),
            hovertemplate='%{location}<br>Clean: %{z:.1f}%<extra></extra>',
        )]
    )
    
    fig.update_geos(**INDIA_GEO)
    fig.update_layout(
        title=dict(text='India Breathes — Clean Energy Map', font=dict(size=14), x=0.5),
        height=500, width=500,
        plot_bgcolor=BG, paper_bgcolor=BG,
        geo=dict(bgcolor=BG),
        margin=dict(l=0, r=0, t=50, b=20),
    )
    return fig

def build_heartbeat():
    """Mirrored stacked area — fossil down, clean up — with a moving cursor."""
    dates = df['date'].values
    
    fig = go.FigureWidget()
    
    fossil_labels = ['Coal', 'Lignite', 'Gas']
    clean_labels = ['Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']
    
    # Fossil traces (negative direction)
    for i, (label, color) in enumerate(zip(fossil_labels, fossil_colors)):
        fig.add_trace(go.Scatter(
            x=dates, y=-fossil_stack[:, i+1],
            fill='tonexty' if i > 0 else 'tozeroy',
            fillcolor=color, line=dict(width=0, color=color),
            name=label, showlegend=True,
            hoverinfo='skip',
        ))
    
    # Clean traces (positive direction)
    for i, (label, color) in enumerate(zip(clean_labels, clean_colors)):
        fig.add_trace(go.Scatter(
            x=dates, y=clean_stack[:, i+1],
            fill='tonexty' if i > 0 else 'tozeroy',
            fillcolor=color, line=dict(width=0, color=color),
            name=label, showlegend=True,
            hoverinfo='skip',
        ))
    
    # Cursor line (vertical) — this is the last trace, updated by index
    cursor_date = dates[0]
    max_y = max(fossil_stack[:, -1].max(), clean_stack[:, -1].max()) * 1.1
    fig.add_trace(go.Scatter(
        x=[cursor_date, cursor_date],
        y=[-max_y, max_y],
        mode='lines',
        line=dict(color='#333', width=2, dash='dot'),
        showlegend=False,
        hoverinfo='skip',
    ))
    
    fig.add_hline(y=0, line_color='#333', line_width=1)
    
    fig.update_layout(
        title=dict(text='The Breathing Grid — Fossil ↓ Clean ↑', font=dict(size=14), x=0.5),
        xaxis=dict(range=[dates[0], dates[-1]]),
        yaxis=dict(range=[-max_y, max_y], title='MU/day', zeroline=True),
        plot_bgcolor=BG, paper_bgcolor=BG,
        height=300, width=1020,
        legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center', font=dict(size=9)),
        margin=dict(l=60, r=20, t=50, b=60),
    )
    return fig

def build_heatmap():
    """Weekly intensity heatmap — 52×7 grid, progressive reveal."""
    fig = go.FigureWidget(
        data=[go.Heatmap(
            z=heat_frames[0],
            x=dow_order,
            y=[f'W{w}' for w in all_weeks],
            colorscale='YlOrRd',
            zmin=heat_z_min, zmax=heat_z_max,
            hovertemplate='Week %{y}, %{x}: %{z:.0f} MU<extra></extra>',
            colorbar=dict(title='MU/day', len=0.6, thickness=15),
        )]
    )
    
    fig.update_layout(
        title=dict(text='Weekly Intensity', font=dict(size=14), x=0.5),
        xaxis_title='',
        yaxis=dict(autorange='reversed', title=''),
        plot_bgcolor=BG, paper_bgcolor=BG,
        height=500, width=300,
        margin=dict(l=40, r=10, t=50, b=20),
    )
    return fig

def build_co2_panel():
    """Cumulative CO2 emissions counter + daily intensity line."""
    fig = go.FigureWidget()
    
    # Cumulative CO2 area (show full year, fill reveals progressively)
    fig.add_trace(go.Scatter(
        x=df['date'].values[:1], y=co2_cumulative[:1],
        fill='tozeroy', fillcolor='rgba(192, 57, 43, 0.3)',
        line=dict(color='#C0392B', width=2),
        name='Cumulative CO₂ (Mt)',
    ))
    
    # Emissions intensity line (secondary y-axis via separate trace)
    fig.add_trace(go.Scatter(
        x=df['date'].values[:1], y=emissions_intensity[:1],
        line=dict(color='#E67E22', width=1.5, dash='dot'),
        name='Intensity (tCO₂/GWh)',
        yaxis='y2',
    ))
    
    max_cum = co2_cumulative[-1] * 1.05
    
    fig.update_layout(
        title=dict(text='CO₂ Emissions — Cumulative + Intensity', font=dict(size=13), x=0.5),
        xaxis=dict(range=[df['date'].min(), df['date'].max()]),
        yaxis=dict(title='Cumulative Mt CO₂', range=[0, max_cum], side='left'),
        yaxis2=dict(title='tCO₂/GWh', overlaying='y', side='right',
                    range=[500, 850], showgrid=False),
        plot_bgcolor=BG, paper_bgcolor=BG,
        height=250, width=510,
        legend=dict(orientation='h', y=-0.3, x=0.5, xanchor='center', font=dict(size=9)),
        margin=dict(l=60, r=60, t=40, b=50),
    )
    return fig

def build_temp_panel():
    """Temperature overlay — India 5-city average with demand correlation."""
    fig = go.FigureWidget()
    
    # Temperature area
    fig.add_trace(go.Scatter(
        x=df['date'].values, y=temp_daily,
        fill='tozeroy', fillcolor='rgba(231, 76, 60, 0.15)',
        line=dict(color='#E74C3C', width=1.5),
        name='Avg Temp (°C)',
    ))
    
    # Total demand on secondary axis
    fig.add_trace(go.Scatter(
        x=df['date'].values, y=df['total'].values,
        line=dict(color='#2980B9', width=1.5, dash='dot'),
        name='Total Demand (MU)',
        yaxis='y2', opacity=0.7,
    ))
    
    # Cursor line
    fig.add_trace(go.Scatter(
        x=[df['date'].values[0], df['date'].values[0]],
        y=[10, 40],
        mode='lines', line=dict(color='#333', width=2, dash='dot'),
        showlegend=False, hoverinfo='skip',
    ))
    
    fig.update_layout(
        title=dict(text='Temperature vs Demand — The AC Effect', font=dict(size=13), x=0.5),
        xaxis=dict(range=[df['date'].min(), df['date'].max()]),
        yaxis=dict(title='°C', range=[10, 40], side='left'),
        yaxis2=dict(title='MU/day', overlaying='y', side='right',
                    range=[df['total'].min()*0.9, df['total'].max()*1.1], showgrid=False),
        plot_bgcolor=BG, paper_bgcolor=BG,
        height=250, width=510,
        legend=dict(orientation='h', y=-0.3, x=0.5, xanchor='center', font=dict(size=9)),
        margin=dict(l=50, r=60, t=40, b=50),
    )
    return fig

print('Build functions ready: build_bloom(), build_map(), build_heartbeat(), build_heatmap(), build_co2_panel(), build_temp_panel()')

Build functions ready: build_bloom(), build_map(), build_heartbeat(), build_heatmap(), build_co2_panel(), build_temp_panel()


---
## Section 3: Update Functions & Master Controller

Each `update_*` function takes a week index (0-51) and swaps data in-place on the FigureWidget. The `master_update` callback fires on every slider tick and calls all four.

In [26]:
def update_bloom(fig, w):
    """Reveal days up to week w."""
    fig.data[0].r = bloom_frames[w]

def update_map(fig, w):
    """Update choropleth to the month corresponding to week w."""
    m = week_to_month(w)
    fig.data[0].z = map_z_frames[m]

def update_heartbeat(fig, w):
    """Move the cursor line to the current week position."""
    day = min((w + 1) * 7, n_days - 1)
    cursor_date = df['date'].iloc[day]
    # The cursor is the last trace (index = n_fossil + n_clean = 8)
    n_traces = len(fig.data)
    fig.data[n_traces - 1].x = [cursor_date, cursor_date]

def update_heatmap(fig, w):
    """Reveal weeks up to w."""
    fig.data[0].z = heat_frames[w]

def update_co2(fig, w):
    """Reveal cumulative CO2 and intensity up to current week."""
    day = min((w + 1) * 7, n_days)
    fig.data[0].x = df['date'].values[:day]
    fig.data[0].y = co2_cumulative[:day]
    fig.data[1].x = df['date'].values[:day]
    fig.data[1].y = emissions_intensity[:day]

def update_temp(fig, w):
    """Move cursor on temperature panel."""
    day = min((w + 1) * 7, n_days - 1)
    cursor_date = df['date'].iloc[day]
    fig.data[2].x = [cursor_date, cursor_date]

print('Update functions ready (6 panels)')

Update functions ready (6 panels)


---
## Section 4: Assembly & Display

Build all widgets, wire up the shared Play + Slider, and display the synchronized dashboard.

**Layout:**
```
[  ▶ Play  |====== Week Slider ======|  Mar 15, 2024  ]
[       Radial Bloom       |       India Map       | Heatmap ]
[              Heartbeat Strip (full width)                  ]
```

In [27]:
# ---------- Build all FigureWidgets ----------
bloom_fig = build_bloom()
map_fig = build_map()
heartbeat_fig = build_heartbeat()
heatmap_fig = build_heatmap()
co2_fig = build_co2_panel()
temp_fig = build_temp_panel()

# ---------- Controls ----------
play = widgets.Play(
    value=0, min=0, max=N_WEEKS - 1,
    step=1, interval=400,
    description='',
    disabled=False,
)

slider = widgets.IntSlider(
    value=0, min=0, max=N_WEEKS - 1,
    step=1, description='Week:',
    continuous_update=True,
    layout=widgets.Layout(width='500px'),
)

date_label = widgets.HTML(
    value=f'<b style="font-size:16px; color:#333;">{week_labels[0]}</b>',
    layout=widgets.Layout(width='120px'),
)

month_label = widgets.HTML(
    value=f'<span style="font-size:13px; color:#666;">Jan 2024</span>',
    layout=widgets.Layout(width='100px'),
)

stats_label = widgets.HTML(value='', layout=widgets.Layout(width='350px'))

widgets.jslink((play, 'value'), (slider, 'value'))

# ---------- Master update callback ----------
def master_update(change):
    w = change['new']
    day = min((w + 1) * 7 - 1, n_days - 1)
    
    with bloom_fig.batch_update():
        update_bloom(bloom_fig, w)
    with map_fig.batch_update():
        update_map(map_fig, w)
    with heartbeat_fig.batch_update():
        update_heartbeat(heartbeat_fig, w)
    with heatmap_fig.batch_update():
        update_heatmap(heatmap_fig, w)
    with co2_fig.batch_update():
        update_co2(co2_fig, w)
    with temp_fig.batch_update():
        update_temp(temp_fig, w)
    
    # Update labels
    date_label.value = f'<b style="font-size:16px; color:#333;">{week_labels[w]}</b>'
    m = week_to_month(w)
    month_label.value = f'<span style="font-size:13px; color:#666;">{month_names[m]} 2024</span>'
    
    cs = df['clean_share'].iloc[day]
    temp = temp_daily[day]
    cum_co2 = co2_cumulative[day]
    cs_color = '#27AE60' if cs > 25 else '#E74C3C'
    stats_label.value = (
        f'<span style="font-size:12px;">'
        f'Clean: <b style="color:{cs_color};">{cs:.1f}%</b> · '
        f'Temp: <b>{temp:.1f}°C</b> · '
        f'CO₂: <b style="color:#C0392B;">{cum_co2:.0f} Mt</b>'
        f'</span>'
    )

slider.observe(master_update, names='value')

# ---------- Layout ----------
title_html = widgets.HTML(
    value='<h2 style="text-align:center; color:#333; margin:5px 0; font-family:Georgia,serif;">'
          'India Breathes — 2024 Electricity Grid, Day by Day</h2>'
          '<p style="text-align:center; color:#888; margin:0; font-size:12px;">'
          'Data: CEA Daily Generation (Robbie Andrew) · POSOCO Regional · Open-Meteo Temperature</p>',
)

controls = widgets.HBox(
    [play, slider, date_label, month_label, stats_label],
    layout=widgets.Layout(justify_content='center', margin='10px 0'),
)

top_row = widgets.HBox(
    [bloom_fig, map_fig, heatmap_fig],
    layout=widgets.Layout(justify_content='center'),
)

mid_row = widgets.HBox(
    [co2_fig, temp_fig],
    layout=widgets.Layout(justify_content='center'),
)

bottom_row = widgets.HBox(
    [heartbeat_fig],
    layout=widgets.Layout(justify_content='center'),
)

dashboard = widgets.VBox(
    [title_html, controls, top_row, mid_row, bottom_row],
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='10px',
        background_color=BG,
    ),
)

display(dashboard)
print('Hit ▶ to watch India breathe through 2024. 6 synchronized panels.')
print('Drag the slider to scrub manually.')

Hit ▶ to watch India breathe through 2024. 6 synchronized panels.
Drag the slider to scrub manually.


---
## Section 5: Static Snapshot Export

Capture the current state of all panels as a high-res composite image. Useful for poster prints or sharing a specific moment (e.g., peak monsoon at week 30).

In [13]:
def export_snapshot(week_idx, filename='india_breathes_snapshot.png'):
    """Export current state as high-res PNG using Kaleido."""
    import kaleido  # ensure installed
    
    # Set the state to the desired week
    slider.value = week_idx
    
    out_dir = '../art/output'
    
    # Export each panel
    bloom_fig.write_image(f'{out_dir}/snapshot_bloom_w{week_idx}.png', scale=2)
    map_fig.write_image(f'{out_dir}/snapshot_map_w{week_idx}.png', scale=2)
    heartbeat_fig.write_image(f'{out_dir}/snapshot_heartbeat_w{week_idx}.png', scale=2)
    heatmap_fig.write_image(f'{out_dir}/snapshot_heatmap_w{week_idx}.png', scale=2)
    
    print(f'Exported 4 panels at week {week_idx} ({week_labels[week_idx]}) to {out_dir}/')

# Example: export peak monsoon (week 30 ≈ late July)
# export_snapshot(30)
print('Run export_snapshot(week_idx) to save PNGs. Example: export_snapshot(30) for peak monsoon.')

Run export_snapshot(week_idx) to save PNGs. Example: export_snapshot(30) for peak monsoon.


---
## Section 6: Video/GIF Export (Matplotlib Path)

ipywidgets are live kernel objects — they can't be "recorded." For video/GIF export, we rebuild the composite as a matplotlib figure and use `FuncAnimation` to step through the same 52 weeks.

**Run this section standalone** after you've explored the interactive version above.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.dates as mdates
from matplotlib.colorbar import ColorbarBase
import geopandas as gpd

plt.rcParams['font.family'] = 'DejaVu Serif'

rdylgn = plt.cm.RdYlGn
bloom_norm = plt.Normalize(vmin=vmin, vmax=vmax)
heat_norm = plt.Normalize(vmin=heat_z_min, vmax=heat_z_max)
map_cmap = plt.cm.RdYlGn
map_norm = plt.Normalize(vmin=0, vmax=80)

gdf = gpd.read_file('../data/raw/india_states.geojson')
gdf['region'] = gdf['ST_NM'].map(state_to_region)

def render_frame(w, axes, artists):
    ax_bloom, ax_map, ax_heat, ax_hb, ax_co2, ax_temp = axes
    day_end = min((w + 1) * 7, n_days)
    m = week_to_month(w)
    
    # ----- Bloom -----
    ax_bloom.clear()
    ax_bloom.set_facecolor(BG)
    angles_rad = np.deg2rad(angles_deg[:day_end])
    widths_rad = np.deg2rad(np.full(day_end, width_deg))
    colors_bloom = [rdylgn(bloom_norm(clean_pct[i])) for i in range(day_end)]
    ax_bloom.bar(angles_rad, bloom_heights[:day_end], width=widths_rad,
                 bottom=1.0, color=colors_bloom, edgecolor='none', alpha=0.9)
    ax_bloom.set_ylim(0, 7)
    ax_bloom.set_xticks([]); ax_bloom.set_yticks([])
    ax_bloom.set_title('Radial Bloom', fontsize=11, fontweight='bold', pad=10)
    for ds, lb in zip([0,31,60,91,121,152,182,213,244,274,305,335], month_names):
        a = (ds / n_days) * 2 * np.pi
        ax_bloom.text(a, 7.8, lb, ha='center', va='center', fontsize=6, color='#666')
    
    # ----- India Map -----
    ax_map.clear()
    ax_map.set_facecolor(BG)
    gdf_f = gdf.copy()
    gdf_f['clean_pct'] = gdf_f['region'].map(lambda r: region_monthly_clean.get((r, m+1), np.nan))
    gdf_f[gdf_f['clean_pct'].notna()].plot(
        ax=ax_map, column='clean_pct', cmap='RdYlGn', vmin=0, vmax=80,
        edgecolor='#aaa', linewidth=0.3)
    gdf_f[gdf_f['clean_pct'].isna()].plot(ax=ax_map, color='#ddd', edgecolor='#aaa', linewidth=0.3)
    ax_map.set_xlim(68, 98); ax_map.set_ylim(6, 38)
    ax_map.set_xticks([]); ax_map.set_yticks([])
    ax_map.set_title(f'Clean Energy — {month_names[m]}', fontsize=11, fontweight='bold', pad=10)
    ax_map.set_aspect('equal')
    
    # ----- Heatmap -----
    ax_heat.clear()
    ax_heat.set_facecolor(BG)
    ax_heat.imshow(heat_frames[w], aspect='auto', cmap='YlOrRd',
                   vmin=heat_z_min, vmax=heat_z_max, interpolation='nearest')
    ax_heat.set_xticks(range(7))
    ax_heat.set_xticklabels(['M','T','W','T','F','S','S'], fontsize=6)
    ax_heat.set_yticks(range(0, len(all_weeks), 10))
    ax_heat.set_yticklabels([f'W{all_weeks[i]}' for i in range(0, len(all_weeks), 10)], fontsize=5)
    ax_heat.set_title('Weekly Intensity', fontsize=11, fontweight='bold', pad=10)
    
    # ----- CO2 Panel -----
    ax_co2.clear()
    ax_co2.set_facecolor(BG)
    dates_mpl = df['date'].values
    ax_co2.fill_between(dates_mpl[:day_end], 0, co2_cumulative[:day_end],
                        color='#C0392B', alpha=0.3)
    ax_co2.plot(dates_mpl[:day_end], co2_cumulative[:day_end], color='#C0392B', linewidth=1.5)
    ax_co2.set_xlim(dates_mpl[0], dates_mpl[-1])
    ax_co2.set_ylim(0, co2_cumulative[-1] * 1.05)
    ax_co2.set_ylabel('Mt CO₂', fontsize=8)
    ax_co2.set_title('Cumulative CO₂', fontsize=11, fontweight='bold', pad=8)
    ax_co2.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax_co2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax_co2.tick_params(axis='both', labelsize=7)
    # Show current value
    if day_end > 1:
        ax_co2.text(0.95, 0.85, f'{co2_cumulative[day_end-1]:.0f} Mt',
                    transform=ax_co2.transAxes, ha='right', fontsize=10,
                    fontweight='bold', color='#C0392B')
    
    # ----- Temperature Panel -----
    ax_temp.clear()
    ax_temp.set_facecolor(BG)
    ax_temp.fill_between(dates_mpl, temp_daily, 10, color='#E74C3C', alpha=0.1)
    ax_temp.plot(dates_mpl, temp_daily, color='#E74C3C', linewidth=1, alpha=0.7, label='Temp (°C)')
    ax_temp_demand = ax_temp.twinx()
    ax_temp_demand.plot(dates_mpl, df['total'].values, color='#2980B9', linewidth=1,
                        alpha=0.6, linestyle='--', label='Demand (MU)')
    ax_temp.axvline(dates_mpl[min(day_end-1, n_days-1)], color='#333', linewidth=1.5, linestyle='--', alpha=0.5)
    ax_temp.set_xlim(dates_mpl[0], dates_mpl[-1])
    ax_temp.set_ylim(10, 40)
    ax_temp.set_ylabel('°C', fontsize=8, color='#E74C3C')
    ax_temp_demand.set_ylabel('MU/day', fontsize=8, color='#2980B9')
    ax_temp.set_title('Temperature vs Demand', fontsize=11, fontweight='bold', pad=8)
    ax_temp.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax_temp.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax_temp.tick_params(axis='both', labelsize=7)
    ax_temp_demand.tick_params(axis='y', labelsize=7)
    # Temp value
    if day_end > 1:
        t = temp_daily[min(day_end-1, n_days-1)]
        ax_temp.text(0.95, 0.85, f'{t:.1f}°C', transform=ax_temp.transAxes,
                     ha='right', fontsize=10, fontweight='bold', color='#E74C3C')
    
    # ----- Heartbeat -----
    ax_hb.clear()
    ax_hb.set_facecolor(BG)
    fossil_labels_list = ['Coal', 'Lignite', 'Gas']
    clean_labels_list = ['Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']
    for i, (label, color) in enumerate(zip(fossil_labels_list, fossil_colors)):
        ax_hb.fill_between(dates_mpl, -fossil_stack[:, i], -fossil_stack[:, i+1],
                           color=color, label=label, linewidth=0)
    for i, (label, color) in enumerate(zip(clean_labels_list, clean_colors)):
        ax_hb.fill_between(dates_mpl, clean_stack[:, i], clean_stack[:, i+1],
                           color=color, label=label, linewidth=0)
    cursor_day = min(day_end - 1, n_days - 1)
    max_y = max(fossil_stack[:, -1].max(), clean_stack[:, -1].max()) * 1.1
    ax_hb.axvline(dates_mpl[cursor_day], color='#333', linewidth=1.5, linestyle='--', alpha=0.7)
    ax_hb.axhline(0, color='#333', linewidth=0.8)
    ax_hb.set_ylim(-max_y, max_y)
    ax_hb.set_ylabel('MU/day', fontsize=8)
    ax_hb.set_title('Fossil ↓  Clean ↑', fontsize=11, fontweight='bold', pad=8)
    ax_hb.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax_hb.xaxis.set_major_locator(mdates.MonthLocator())
    ax_hb.tick_params(axis='both', labelsize=7)
    ax_hb.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=8, fontsize=7, frameon=False)
    
    return artists

print('Render function ready — 6 panels with colorbars, legends, CO2, temperature.')


In [ ]:
%%time

# 6-panel layout:
# Row 1: [bloom] [map] [heatmap]
# Row 2: [CO2]  [temperature]
# Row 3: [heartbeat — full width]

fig_export = plt.figure(figsize=(16, 16))
fig_export.set_facecolor(BG)

# Use GridSpec for precise control
gs = fig_export.add_gridspec(4, 6, height_ratios=[1.3, 0.55, 0.55, 0.05],
                              width_ratios=[1.2, 0.05, 1.2, 0.05, 0.5, 0.05],
                              hspace=0.35, wspace=0.3)

ax_bloom = fig_export.add_subplot(gs[0, 0], projection='polar')
ax_bloom_cb = fig_export.add_subplot(gs[0, 1])  # colorbar
ax_map = fig_export.add_subplot(gs[0, 2])
ax_map_cb = fig_export.add_subplot(gs[0, 3])  # colorbar
ax_heat = fig_export.add_subplot(gs[0, 4])
ax_heat_cb = fig_export.add_subplot(gs[0, 5])  # colorbar
ax_co2 = fig_export.add_subplot(gs[1, 0:3])
ax_temp = fig_export.add_subplot(gs[1, 3:6])
ax_hb = fig_export.add_subplot(gs[2, :])

# Static colorbars
from matplotlib.colorbar import ColorbarBase
cb1 = ColorbarBase(ax_bloom_cb, cmap=plt.cm.RdYlGn,
                   norm=plt.Normalize(vmin=vmin, vmax=vmax))
cb1.set_label('Clean %', fontsize=7)
cb1.ax.tick_params(labelsize=6)

cb2 = ColorbarBase(ax_map_cb, cmap=plt.cm.RdYlGn,
                   norm=plt.Normalize(vmin=0, vmax=80))
cb2.set_label('Clean %', fontsize=7)
cb2.ax.tick_params(labelsize=6)

cb3 = ColorbarBase(ax_heat_cb, cmap=plt.cm.YlOrRd,
                   norm=plt.Normalize(vmin=heat_z_min, vmax=heat_z_max))
cb3.set_label('MU/day', fontsize=7)
cb3.ax.tick_params(labelsize=6)

# Credit line
fig_export.text(0.5, 0.01, 'Data: CEA (Robbie Andrew) · POSOCO · Open-Meteo  |  One Sensor, One Year — Edition 1',
                ha='center', fontsize=9, color='#999', style='italic')

# Title
fig_export.suptitle('India Breathes — 2024 Electricity Grid',
                     fontsize=22, fontweight='bold', color='#333',
                     fontfamily='DejaVu Serif', y=0.98)

date_text = fig_export.text(0.5, 0.95, '', ha='center', fontsize=15, color='#555')

def animate(w):
    date_text.set_text(f'{week_labels[w]}  |  {month_names[week_to_month(w)]} 2024')
    render_frame(w, (ax_bloom, ax_map, ax_heat, ax_hb, ax_co2, ax_temp), [])
    return []

anim = FuncAnimation(fig_export, animate, frames=N_WEEKS, interval=400, blit=False)

gif_path = '../art/output/india_breathes_2024.gif'
anim.save(gif_path, writer=PillowWriter(fps=3), dpi=100)
plt.close(fig_export)

print(f'GIF saved to {gif_path}')
print(f'52 frames at 3 fps = ~17 second loop')
print(f'6 panels: bloom, map, heatmap (with colorbars) + CO2, temperature, heartbeat (with legend)')


---
## What's Here & What's Next

**This notebook gives you:**
- 4 synchronized animated panels driven by one shared clock
- Interactive Play/Pause + manual scrub in Jupyter
- Date, month, and live clean share readout
- Static PNG snapshot export (via Kaleido)
- GIF export (via matplotlib FuncAnimation)

**Exploration ideas:**
- Adjust `play.interval` (ms) to speed up/slow down the animation
- Try different color scales on the bloom (e.g., `'Viridis'`, `'Turbo'`)
- Add a 5th panel — daily clean share line chart? Cumulative emissions counter?
- Swap the heatmap for a donut chart showing fuel mix at the current week

**For the final web version:**
- Export FigureWidget JSON via `fig.to_json()` → load in Plotly.js
- Replace ipywidgets with a JavaScript `requestAnimationFrame` loop
- Eventually rewrite the bloom in D3.js for glow effects and smoother transitions